# C.E.L.I.A. · Treino do modelo de buracos [HU-06, RF09]

Treina um YOLOv8n para detectar buracos. Saida: `buracos.pt`, usado pela API de visao.

**Dataset:** "Pothole Dataset", por KugaN, Roboflow Universe, versao 2 (1.595 imagens, 1 classe: `pothole`).
Link: https://universe.roboflow.com/kugan/pothole-mftcc
Licenca: CC BY 4.0 (uso permitido com citacao da fonte).

**Antes de rodar:**
1. Menu *Ambiente de execucao > Alterar tipo de ambiente* e escolha **GPU T4**.
2. No icone de chave (Secrets), na barra lateral, crie o segredo `ROBOFLOW_API_KEY` com a sua chave do Roboflow e ative o acesso do notebook.
   Nunca cole a chave direto no codigo nem suba para o GitHub.

## 1. Conferir se a GPU esta ligada

In [ ]:
# Se aparecer uma tabela com "Tesla T4", a GPU esta ativa.
!nvidia-smi

## 2. Instalar as bibliotecas

In [ ]:
# ultralytics: biblioteca do YOLO (treino e deteccao)
# roboflow: baixa o dataset direto do Roboflow
!pip install -q ultralytics roboflow

## 3. Onde salvar os resultados

- `USAR_DRIVE = True`: salva no Google Drive (seguro se o Colab desligar). Precisa de espaco livre no Drive.
- `USAR_DRIVE = False`: salva so no Colab e, no fim, baixa o `buracos.pt` para o seu computador. Use se o Drive estiver cheio.

In [ ]:
import os

USAR_DRIVE = False  # troque para True se tiver espaco no Drive

if USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_DRIVE = '/content/drive/MyDrive/celia_visao'
else:
    PASTA_DRIVE = '/content/celia_visao'  # pasta temporaria do Colab: some quando a sessao fecha

os.makedirs(PASTA_DRIVE, exist_ok=True)
print('Resultados serao salvos em:', PASTA_DRIVE)

## 4. Baixar o dataset no formato YOLOv8

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

chave = userdata.get('ROBOFLOW_API_KEY')  # le a chave do Secrets, sem expor no codigo
rf = Roboflow(api_key=chave)
projeto = rf.workspace('kugan').project('pothole-mftcc')
dataset = projeto.version(2).download('yolov8')

print('Dataset baixado em:', dataset.location)

## 5. Ajustar o `data.yaml`

O arquivo do Roboflow usa caminhos relativos que as vezes confundem o YOLO.
Aqui reescrevemos com caminhos completos.

In [ ]:
import yaml

caminho_yaml = f'{dataset.location}/data.yaml'
with open(caminho_yaml) as arquivo:
    config = yaml.safe_load(arquivo)

config['train'] = f'{dataset.location}/train/images'
config['val'] = f'{dataset.location}/valid/images'
config['test'] = f'{dataset.location}/test/images'

with open(caminho_yaml, 'w') as arquivo:
    yaml.safe_dump(config, arquivo)

print(config)  # deve mostrar nc: 1 e names: ['pothole']

## 5b. Adicionar imagens negativas (PROPOSTA, ainda nao aprovada)

Imagens negativas sao fotos de calcada **sem buraco** (ex.: calcadao de Copacabana com desenhos de pedra preta).
Elas ensinam o modelo que mancha escura desenhada nao e buraco (falso positivo do teste de 12/11).

Nao precisa marcar nada no Roboflow: para o YOLO, uma imagem na pasta de treino **sem arquivo de rotulo** e uma imagem de fundo ("nada aqui").

**Como usar:** coloque as fotos numa pasta, compacte como `negativas.zip` e envie quando a celula pedir.
Fontes e licencas das fotos: `visao/treino/fontes_negativas.md`. Nao coloque aqui as fotos usadas no teste da API.
Para pular esta etapa, deixe `USAR_NEGATIVAS = False`.

In [ ]:
import zipfile, shutil, glob

USAR_NEGATIVAS = True

if USAR_NEGATIVAS:
    from google.colab import files
    enviado = files.upload()  # escolha o negativas.zip no seu computador
    pasta_tmp = '/content/negativas'
    with zipfile.ZipFile('negativas.zip') as z:
        z.extractall(pasta_tmp)

    pasta_treino_img = f'{dataset.location}/train/images'
    copiadas = 0
    for caminho in glob.glob(f'{pasta_tmp}/**/*', recursive=True):
        if caminho.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
            # prefixo 'neg_' evita conflito de nome com as imagens do dataset
            destino = f'{pasta_treino_img}/neg_{os.path.basename(caminho)}'
            shutil.copy(caminho, destino)  # sem .txt de rotulo = imagem de fundo
            copiadas += 1
    print(f'{copiadas} imagens negativas adicionadas ao treino.')
    print('Total de imagens de treino:', len(glob.glob(f'{pasta_treino_img}/*')))


## 6. Treinar

- `yolov8n.pt`: versao *nano*, a menor e mais rapida do YOLOv8, ja pre-treinada. Nos so ensinamos a ela o que e buraco (*transfer learning*).
- `epochs=50`: o modelo ve o dataset inteiro 50 vezes. Na T4 leva cerca de 20 a 40 minutos.
- `patience=15`: para antes se passar 15 epocas sem melhorar.

In [ ]:
from ultralytics import YOLO

modelo = YOLO('yolov8n.pt')
resultado_treino = modelo.train(
    data=caminho_yaml,
    epochs=50,
    imgsz=640,
    patience=15,
    project=PASTA_DRIVE,
    name='treino_buracos',
    exist_ok=True,
)

## 7. Avaliar no conjunto de teste (imagens que o modelo nunca viu)

Referencia: o modelo publicado junto com o dataset tem mAP@50 de 73,3%, precisao de 77,3% e recall de 64,4%.
- **Precisao:** das caixas que o modelo desenhou, quantas eram buraco de verdade.
- **Recall:** dos buracos que existiam, quantos o modelo encontrou.
- **mAP@50:** nota geral de deteccao (quanto maior, melhor).

In [ ]:
pasta_treino = f'{PASTA_DRIVE}/treino_buracos'
melhor = YOLO(f'{pasta_treino}/weights/best.pt')
metricas = melhor.val(data=caminho_yaml, split='test')

print(f'mAP@50:   {metricas.box.map50:.1%}')
print(f'Precisao: {metricas.box.mp:.1%}')
print(f'Recall:   {metricas.box.mr:.1%}')

## 8. Testar em algumas imagens de teste

In [ ]:
import glob
from IPython.display import Image, display

imagens_teste = sorted(glob.glob(f'{dataset.location}/test/images/*'))[:4]
resultados = melhor.predict(imagens_teste, conf=0.4, save=True,
                            project=PASTA_DRIVE, name='previsoes', exist_ok=True)

for r in resultados:
    caminho_salvo = f'{r.save_dir}/{os.path.basename(r.path)}'
    display(Image(filename=caminho_salvo, width=500))

## 9. Salvar como `buracos.pt` e baixar

In [ ]:
import shutil

destino = f'{PASTA_DRIVE}/buracos.pt'
shutil.copy(f'{pasta_treino}/weights/best.pt', destino)
print('Modelo final salvo em:', destino)

if not USAR_DRIVE:
    from google.colab import files
    files.download(destino)  # baixa para o seu computador; aceite o download no navegador

print('Coloque o buracos.pt em visao/api/modelos/ no repositorio.')